In [2]:
# Import libries
import numpy as np
import pandas as pd
import os
import re
import glob
import json
import csv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyranges as pr
import pysam
from matplotlib.patches import Patch

# from scipy.stats import mannwhitneyu, stats
# from statannotations.Annotator import Annotator

current_directory = os.getcwd()
print("Current Directory:", current_directory)
pd.set_option("display.max_columns", None)


Current Directory: /mnt/NAS3/home/jiwon/ECTRES/python


In [3]:
aaSuite_gemline_ms=pd.read_csv('../summary/aaSuite_germline_ms/10X/aaSuite_gemline_ms_all_20260430.csv')


amplicons=pd.read_csv('/mnt/NAS3/home/jiwon/ECTRES/data/merged/amplicons.csv')
samples=pd.read_csv('/mnt/NAS3/home/jiwon/ECTRES/data/merged/samples.csv')


In [4]:
aaSuite_gemline_ms.head()
check=['EG_X01','NCI_X01']
aaSuite_gemline_ms[aaSuite_gemline_ms['sample_id'].isin(check)]['aa_barcode'].unique()

array(['ECTRES-H2170-0001-TPX-A10-WGS-8CY686',
       'ECTRES-ECGI1-0001-TPX-A21-WGS-1UM757'], dtype=object)

In [5]:
samples.columns

Index(['aa_barcode', 'cellline_name', 'sample_type', 'drug_type',
       'bulk_or_singlecc'],
      dtype='object')

In [6]:
samples[samples['aa_barcode'].isin(['ECTRES-H2170-0001-TPX-A10-WGS-8CY686',
       'ECTRES-ECGI1-0001-TPX-A21-WGS-1UM757'])]

,aa_barcode,cellline_name,sample_type,drug_type,bulk_or_singlecc
63,ECTRES-ECGI1-0001-TPX-A21-WGS-1UM757,EC-GI-10,Sensitive,na,singlecc
97,ECTRES-H2170-0001-TPX-A10-WGS-8CY686,NCI-H2170,Sensitive,na,singlecc


In [7]:
columns = ['aa_barcode', 'matched_mismatched','new_aa_barcode','new_cellline_name','new_sample_type','new_drug_type','new_bulk_or_singlecc']
df = samples.copy()

df['new_aa_barcode'] = df['aa_barcode']
df['new_sample_type'] = df['sample_type']
df['new_drug_type'] = df['drug_type']
df['new_bulk_or_singlecc'] = df['bulk_or_singlecc']

# 2. new_cellline_name 조건부 할당
# 일단 전체를 기존 cellline_name으로 채워둡니다.
df['new_cellline_name'] = df['cellline_name']

# 특정 바코드에 해당하는 행만 값을 변경합니다.
df.loc[df['aa_barcode'] == 'ECTRES-ECGI1-0001-TPX-A21-WGS-1UM757', 'new_cellline_name'] = 'NCI-H2170'
df.loc[df['aa_barcode'] == 'ECTRES-H2170-0001-TPX-A10-WGS-8CY686', 'new_cellline_name'] = 'EC-GI-10'

# 3. 결과 확인 (잘 바뀌었는지 해당 샘플들만 확인)
df[df['aa_barcode'].isin(['ECTRES-ECGI1-0001-TPX-A21-WGS-1UM757', 'ECTRES-H2170-0001-TPX-A10-WGS-8CY686'])]


mismatch_list = [
    'ECTRES-ECGI1-0001-TPX-A21-WGS-1UM757', 
    'ECTRES-H2170-0001-TPX-A10-WGS-8CY686'
]

df['matched_mismatched'] = np.where(df['aa_barcode'].isin(mismatch_list), 'mismatched', 'matched')

# 최종 결과 확인용 (원하는 컬럼 순서대로 출력)
df[columns].head()



,aa_barcode,matched_mismatched,new_aa_barcode,new_cellline_name,new_sample_type,new_drug_type,new_bulk_or_singlecc
0,DRUGBR-A549X-0001-TPX-M01-WGS-2JX855,matched,DRUGBR-A549X-0001-TPX-M01-WGS-2JX855,A549,Sensitive,na,bulk
1,DRUGBR-A549X-0001-TPX-M02-WGS-3MF090,matched,DRUGBR-A549X-0001-TPX-M02-WGS-3MF090,A549,Resistant,Cisplatin,bulk
2,DRUGBR-BT20L-0001-TPX-M01-WGS-2AD667,matched,DRUGBR-BT20L-0001-TPX-M01-WGS-2AD667,BT-20 LUC,Sensitive,na,bulk
3,DRUGBR-BT20L-0001-TPX-M02-WGS-4FF385,matched,DRUGBR-BT20L-0001-TPX-M02-WGS-4FF385,BT-20 LUC,Resistant,Erlotinib,bulk
4,DRUGBR-BT474-0001-TPX-C02-WGS-6UC589,matched,DRUGBR-BT474-0001-TPX-C02-WGS-6UC589,BT-474 LUC,Resistant,Doxorubicin,bulk


In [8]:
df[df['aa_barcode'].isin(['ECTRES-ECGI1-0001-TPX-A21-WGS-1UM757', 'ECTRES-H2170-0001-TPX-A10-WGS-8CY686'])][columns]


,aa_barcode,matched_mismatched,new_aa_barcode,new_cellline_name,new_sample_type,new_drug_type,new_bulk_or_singlecc
63,ECTRES-ECGI1-0001-TPX-A21-WGS-1UM757,mismatched,ECTRES-ECGI1-0001-TPX-A21-WGS-1UM757,NCI-H2170,Sensitive,na,singlecc
97,ECTRES-H2170-0001-TPX-A10-WGS-8CY686,mismatched,ECTRES-H2170-0001-TPX-A10-WGS-8CY686,EC-GI-10,Sensitive,na,singlecc


In [9]:
# df[columns].to_csv('/mnt/NAS3/home/jiwon/ECTRES/data/merged/matched_samples.csv',index=False)

In [10]:
# manifest=pd.read_csv('../manifest/ECTRES_clones_nf_dna_fastqs_20260303.csv')
manifest=pd.read_csv('../manifest/ECTRES_clones_nf_dna_bam.csv')

# 
manifest.head(2)

# manifest['aliquot_barcode'].nunique()

,aliquot_barcode,source_barcode,sample_barcode,patient_barcode,sample_type,tumor_or_normal,sequence_type,sample_legacy_id,gender,action,sample_id,bam,bai
0,ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349,ECGI1,ECTRES-ECGI1-0001-TPX-A01,ECTRES-ECGI1-0001,TP,tumor,WGS,EG_1,XY,NaN,EG_1,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...
1,ECTRES-ECGI1-0001-TPX-A10-WGS-3SW949,ECGI1,ECTRES-ECGI1-0001-TPX-A10,ECTRES-ECGI1-0001,TP,tumor,WGS,EG_10,XY,NaN,EG_10,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...


In [11]:
aaSuite_gemline_ms=pd.read_csv('../summary/aaSuite_germline_ms/10X/aaSuite_gemline_ms_all_20260430.csv')
aaSuite_gemline_ms.head(2)

aaSuite_gemline_ms.columns

aaSuite_gemline_ms[['aa_barcode', 'source_barcode', 'sample_id']].drop_duplicates(keep='first').shape
aa_sub=aaSuite_gemline_ms[['aa_barcode', 'source_barcode', 'sample_id']].drop_duplicates(keep='first')
aa_sub.head(2)

,aa_barcode,source_barcode,sample_id
0,ECTRES-EFM19-0001-TPX-A05-WGS-4GXSY5,EFM19,EFM_5
14,ECTRES-EFM19-0001-TPX-A09-WGS-MS1ULC,EFM19,EFM_9


In [12]:
aa_mani=pd.merge(manifest,aa_sub, left_on='aliquot_barcode',right_on='aa_barcode',how='left')
aa_mani.head(2)

,aliquot_barcode,source_barcode_x,sample_barcode,patient_barcode,sample_type,tumor_or_normal,sequence_type,sample_legacy_id,gender,action,sample_id_x,bam,bai,aa_barcode,source_barcode_y,sample_id_y
0,ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349,ECGI1,ECTRES-ECGI1-0001-TPX-A01,ECTRES-ECGI1-0001,TP,tumor,WGS,EG_1,XY,NaN,EG_1,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349,ECGI1,EG_1
1,ECTRES-ECGI1-0001-TPX-A10-WGS-3SW949,ECGI1,ECTRES-ECGI1-0001-TPX-A10,ECTRES-ECGI1-0001,TP,tumor,WGS,EG_10,XY,NaN,EG_10,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,ECTRES-ECGI1-0001-TPX-A10-WGS-3SW949,ECGI1,EG_10


In [13]:
# manifest_ref=pd.read_csv('/mnt/NAS3/home/jiwon/yam_local/oncoanalyser/manifest/BIOCHP_JW_WGS_oncoanalyser_manifest_fqs.csv')
# print(manifest_ref.shape)
# print(manifest_ref.columns)
# manifest_ref.head(2)

In [14]:
df = aa_mani.copy()
df['group_id']=df['aliquot_barcode']
df['subject_id']=df['source_barcode_y']
df['sample_id']=df['sample_id_y']
df['sample_type']='tumor'
df['sequence_type']='dna'
df['filetype']='bam'
# df['info']='library_id:'+df['sample_id']+';lane:001'
df['filepath']=df['bam']
df['action']='run'

# df[['group_id', 'subject_id', 'sample_id', 'sample_type', 'sequence_type','filetype', 'filepath', 'action']].to_csv('../manifest/ECTRES_clones_nf_dna_bam_oncoanalyser.csv',index=False)

df[['group_id', 'subject_id', 'sample_id', 'sample_type', 'sequence_type','filetype', 'filepath', 'action']].head(2)

,group_id,subject_id,sample_id,sample_type,sequence_type,filetype,filepath,action
0,ECTRES-ECGI1-0001-TPX-A01-WGS-6DM349,ECGI1,EG_1,tumor,dna,bam,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,run
1,ECTRES-ECGI1-0001-TPX-A10-WGS-3SW949,ECGI1,EG_10,tumor,dna,bam,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,run


In [15]:
df[df['sample_id']=='parental'][['group_id', 'subject_id', 'sample_id', 'sample_type', 'sequence_type','filetype', 'filepath', 'action']].to_csv('../manifest/ECTRES_clones_nf_dna_bam_oncoanalyser.csv',index=False)
df[df['sample_id']=='parental'][['group_id', 'subject_id', 'sample_id', 'sample_type', 'sequence_type','filetype', 'filepath', 'action']]

,group_id,subject_id,sample_id,sample_type,sequence_type,filetype,filepath,action
50,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,H2170,parental,tumor,dna,bam,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,run
51,ECTRES-ECGI1-0001-TPX-A01-WGS-1ST985,ECGI1,parental,tumor,dna,bam,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,run
52,ECTRES-EFM19-0001-TPX-A01-WGS-2PV977,EFM19,parental,tumor,dna,bam,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,run


In [35]:
df[df['group_id']=='ECTRES-H2170-0001-TPX-A01-WGS-3YV111']
df[df['sample_id']=='EG_X01']

# FQ1 컬럼이 '/mnt/NAS2'로 시작하는 행들만 추출
# df[df['FQ1'].str.startswith('/mnt/NAS2', na=False)]

,aliquot_barcode,source_barcode_x,sample_barcode,patient_barcode,sample_type,tumor_or_normal,sequence_type,sample_legacy_id,gender,action,sample_id_x,bam,bai,aa_barcode,source_barcode_y,sample_id_y,group_id,subject_id,sample_id,filetype,filepath
34,ECTRES-H2170-0001-TPX-A10-WGS-8CY686,H2170,ECTRES-H2170-0001-TPX-A10,ECTRES-H2170-0001,tumor,tumor,dna,NCI_10,XY,run,NCI_10,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,ECTRES-H2170-0001-TPX-A10-WGS-8CY686,ECGI1,EG_X01,ECTRES-H2170-0001-TPX-A10-WGS-8CY686,ECGI1,EG_X01,bam,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...


In [21]:
aaSuite_gemline_ms[aaSuite_gemline_ms['aa_barcode']=='ECTRES-H2170-0001-TPX-A01-WGS-3YV111']

,amplicon_barcode,aa_barcode,amplicon_number,amplicon_decomposition_class,ecDNA+,BFB+,ecDNA_amplicons,AmpliconID,amplicon_index,aa_summary_file_path,N_Intervals,Intervals,OncogenesAmplified,TotalIntervalSize,AmplifiedIntervalSize,AverageAmplifiedCopyCount,N_Chromosomes,N_SequenceEdges,N_BreakpointEdges,N_CoverageShifts,N_MeanshiftSegmentsCopyCount>5,N_Foldbacks,N_CoverageShiftsWithBreakpointEdges,source_barcode,sample_id,amplicon_type,old_source_barcode,old_sample_id
2393,ECTRES-H2170-0001-TPX-A01-WGS-3YV111-amplicon10,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,amplicon10,No amp/Invalid,None detected,None detected,0,10,amplicon10,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,1,7:3432367-3512366,",",80000,79999,2.758400,1,1,0,0,0,0,0,H2170,parental,none,H2170,parental
2394,ECTRES-H2170-0001-TPX-A01-WGS-3YV111-amplicon11,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,amplicon11,No amp/Invalid,None detected,None detected,0,11,amplicon11,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,1,8:13426936-14301994,",",875059,0,2.000000,1,1,0,0,0,0,0,H2170,parental,none,H2170,parental
2395,ECTRES-H2170-0001-TPX-A01-WGS-3YV111-amplicon12,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,amplicon12,Cyclic,Positive,None detected,1,12,amplicon12,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,3,"8:43225919-43227230,14:32583156-32584467,14:34...","NKX2-1,FOXA1,",6007596,6006385,8.620273,2,53,18,11,1,1,10,H2170,parental,ecDNA,H2170,parental
2396,ECTRES-H2170-0001-TPX-A01-WGS-3YV111-amplicon13,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,amplicon13,No amp/Invalid,None detected,None detected,0,13,amplicon13,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,1,8:133741504-136796510,"WISP1,NDRG1,",3055007,0,2.000000,1,1,0,0,0,0,0,H2170,parental,none,H2170,parental
2397,ECTRES-H2170-0001-TPX-A01-WGS-3YV111-amplicon14,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,amplicon14,No amp/Invalid,None detected,None detected,0,14,amplicon14,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,1,9:254991-419988,",",164998,0,2.000000,1,1,0,0,0,0,0,H2170,parental,none,H2170,parental
2398,ECTRES-H2170-0001-TPX-A01-WGS-3YV111-amplicon15,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,amplicon15,No amp/Invalid,None detected,None detected,0,15,amplicon15,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,1,10:57708897-59443914,",",1735018,0,2.000000,1,4,1,0,0,0,0,H2170,parental,none,H2170,parental
2399,ECTRES-H2170-0001-TPX-A01-WGS-3YV111-amplicon16,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,amplicon16,No amp/Invalid,None detected,None detected,0,16,amplicon16,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,1,11:34943503-35683521,",",740019,0,2.000000,1,2,0,0,0,0,0,H2170,parental,none,H2170,parental
2400,ECTRES-H2170-0001-TPX-A01-WGS-3YV111-amplicon17,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,amplicon17,No amp/Invalid,None detected,None detected,0,17,amplicon17,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,1,11:69354762-70359653,"FGF4,CCND1,CTTN,FGF3,",1004892,0,2.000000,1,4,0,2,0,0,2,H2170,parental,none,H2170,parental
2401,ECTRES-H2170-0001-TPX-A01-WGS-3YV111-amplicon18,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,amplicon18,No amp/Invalid,None detected,None detected,0,18,amplicon18,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,1,11:128416697-129836659,"FLI1,PRDM10,KCNJ5,",1419963,0,2.000000,1,2,0,0,0,0,0,H2170,parental,none,H2170,parental
2402,ECTRES-H2170-0001-TPX-A01-WGS-3YV111-amplicon19,ECTRES-H2170-0001-TPX-A01-WGS-3YV111,amplicon19,No amp/Invalid,None detected,None detected,0,19,amplicon19,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,1,14:51419831-54769815,",",3349985,3349982,3.232278,1,3,0,0,0,0,0,H2170,parental,none,H2170,parental


In [40]:
# manifest=pd.read_csv('../manifest/ECTRES_clones_nf_dna_bam.csv')
manifest.head(2)

manifest[manifest['source_barcode']=='H2170'].head(2)

,aliquot_barcode,source_barcode,sample_barcode,patient_barcode,sample_type,tumor_or_normal,sequence_type,sample_legacy_id,gender,action,sample_id,bam,bai
33,ECTRES-H2170-0001-TPX-A01-WGS-6TB808,H2170,ECTRES-H2170-0001-TPX-A01,ECTRES-H2170-0001,TP,tumor,WGS,NCI_1,XY,NaN,NCI_1,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...
34,ECTRES-H2170-0001-TPX-A10-WGS-8CY686,H2170,ECTRES-H2170-0001-TPX-A10,ECTRES-H2170-0001,TP,tumor,WGS,NCI_10,XY,NaN,NCI_10,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...,/mnt/NAS3/home/mary/HL-NF/scratch/ECTRES/resul...


In [41]:
outlier=['EG_1','EG_5','NCI_23','NCI_25','NCI_30']
manifest[manifest['sample_id'].isin(outlier)].to_csv('../manifest/ECTRES_clones_nf_dna_bam_outlier.csv',index=False)

